# Propagating photometric errors into the ensemble p(z): Cardinal Y1 complete

The first task of `CNN_photoz-ensemble-cardinal.ipynb` (noisy Cardinal, Y1 depth, i < 23, all 9 bands), testing a simple way of putting photometric uncertainty into the 1-parameter ensemble's p(z): **Monte Carlo propagation of the photometric errors at prediction time** with `znn.ensemble_predict_resampled`.

For each test object it draws S noise realizations of the photometry from the catalogue's `_err` columns (`znn.resample_photometry`: Gaussian noise in flux; negative flux becomes a non-detection), runs **every** ensemble member on **every** realization, and builds a Gaussian p(z) from the mean and spread of the S × K predictions. The number of realizations S is independent of the number of members K.

The width then has two parts, which add in quadrature:
- **model**: the spread between members on the same photometry (all the baseline has);
- **photometric**: how much the ensemble's prediction moves under the object's own photometric noise. This grows with the errors and with how sensitive the redshift is to the photometry.

Versions compared on the test set:
- **baseline**: the ensemble on the observed photometry (`znn.ensemble_predict`); the width is model spread only;
- **resampled, S = …**: the same trained ensemble with S realizations, for each S in `N_SAMPLES`.

The observed photometry is already one noisy draw, so redrawing it adds the noise a second time. Expect the widths to follow the errors' object-to-object pattern better than their overall level.

The trained ensemble is saved under `MODEL_ROOT` and reloaded on re-runs.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from qp.metrics.pit import PIT

plt.rc("font", size=14)
plt.rc("axes", labelsize=14, titlesize=14)
plt.rc("legend", fontsize=14)
plt.rc("xtick", labelsize=10)
plt.rc("ytick", labelsize=10)

import znn  # run from the repo root so this picks up the local source

## Configuration

In [ ]:
# Parametric paths (edit via environment variables, or defaults below)
CARDINAL_DATA_ROOT = os.environ.get("CNNPZ_CARDINAL_DATA_ROOT", "data")
FILTER_ROOT = os.environ.get("CNNPZ_FILTER_ROOT", "data")
MODEL_ROOT = os.environ.get("CNNPZ_MODEL_ROOT", "models/noise_realizations")

TRAIN_FILE = os.path.join(CARDINAL_DATA_ROOT, "train_100k_noisy_y1_i23.parquet")
TEST_FILE = os.path.join(CARDINAL_DATA_ROOT, "test_100k_noisy_y1_i23.parquet")

# band -> (catalogue magnitude column, filter curve file), ordered by wavelength. The Roman columns
# are read under their original names, so no renaming is needed.
BANDS = {
    "u": ("mag_u_lsst", os.path.join(FILTER_ROOT, "DC2LSST_u.res")),
    "g": ("mag_g_lsst", os.path.join(FILTER_ROOT, "DC2LSST_g.res")),
    "r": ("mag_r_lsst", os.path.join(FILTER_ROOT, "DC2LSST_r.res")),
    "i": ("mag_i_lsst", os.path.join(FILTER_ROOT, "DC2LSST_i.res")),
    "z": ("mag_z_lsst", os.path.join(FILTER_ROOT, "DC2LSST_z.res")),
    "y": ("mag_y_lsst", os.path.join(FILTER_ROOT, "DC2LSST_y.res")),
    "Y": ("Roman_obs_Y106", os.path.join(FILTER_ROOT, "roman_Y106.res")),
    "J": ("Roman_obs_J129", os.path.join(FILTER_ROOT, "roman_J129.res")),
    "H": ("Roman_obs_H158", os.path.join(FILTER_ROOT, "roman_H158.res")),
}
REF_BAND = "i"  # magnitudes are normalized by this band

# magnitude column -> photometric error column, used to draw the noise realizations
ERRORS = {column: f"{column}_err" for column, _ in BANDS.values()}

N_SPLITS = 5  # ensemble members (K-fold)
EPOCHS = 100  # max epochs per member (early stopping usually ends sooner)
N_SAMPLES = [10, 20, 50]  # numbers of noise realizations S to compare
NOISE_SEED = 42

## Load training and test data

In [ ]:
training_y1 = znn.read_catalog(TRAIN_FILE)
test_y1 = znn.read_catalog(TEST_FILE)

X_y1, Y_y1 = znn.catalog_to_XY(training_y1, BANDS, REF_BAND)
X_test_y1, Y_test_y1 = znn.catalog_to_XY(test_y1, BANDS, REF_BAND)
mag_i_test = test_y1["mag_i_lsst"].to_numpy()
print("train:", X_y1.shape, "test:", X_test_y1.shape)

## What one noise realization looks like

How far one realization moves the test magnitudes: in units of the quoted errors (should be a unit Gaussian), and in magnitudes against i.

In [ ]:
example = znn.resample_photometry(test_y1, ERRORS, np.random.default_rng(NOISE_SEED))
fig, axarr = plt.subplots(1, 2, figsize=[13, 4.5])
for column in ["mag_u_lsst", "mag_r_lsst", "mag_y_lsst", "Roman_obs_H158"]:
    mag, err, new = test_y1[column].to_numpy(), test_y1[f"{column}_err"].to_numpy(), example[column].to_numpy()
    ok = np.isfinite(mag) & np.isfinite(new)
    axarr[0].hist(((new - mag) / err)[ok], bins=np.linspace(-5, 5, 101), density=True, histtype="step", label=column)
    axarr[1].scatter(mag_i_test[ok][::20], (new - mag)[ok][::20], s=0.5, label=column)
    print(f"{column}: {np.mean(np.isfinite(mag) & np.isnan(new)):.2%} of detections become non-detections in this realization")
zz = np.linspace(-5, 5, 200)
axarr[0].plot(zz, np.exp(-0.5 * zz**2) / np.sqrt(2 * np.pi), "k--", label="unit Gaussian")
axarr[0].set_xlabel("(resampled - observed) / error")
axarr[0].legend(fontsize=9)
axarr[1].set_xlabel("i magnitude")
axarr[1].set_ylabel("resampled - observed [mag]")
axarr[1].set_ylim(-1.5, 1.5)
axarr[1].legend(fontsize=9, markerscale=10)
plt.tight_layout()
plt.show()

## Train the ensemble (or load if already trained)

In [ ]:
save_dir = os.path.join(MODEL_ROOT, "y1_complete_point_ensemble_CNN_6layers")
if os.path.exists(os.path.join(save_dir, "norm_params.json")):
    trained_models = znn.load_ensemble(save_dir=save_dir)
    histories = None
else:
    trained_models, histories = znn.train_ensembles(znn.build_model, X_y1, Y_y1, N_SPLITS=N_SPLITS, EPOCHS=EPOCHS)
    znn.save_ensemble(trained_models, save_dir=save_dir)

if histories is not None:
    znn.plot_ensemble_losses(histories, ylim=(0, 0.001))

## Predictions on the test set

`width` is the standard deviation of each object's p(z). For the resampled versions, `components` holds its split into model and photometric parts.

In [ ]:
ids = test_y1["galaxy_id"].to_numpy()
pz, components = {}, {}

pz["baseline"] = znn.ensemble_predict(trained_models, X_test_y1, ids=ids)
for n_samples in N_SAMPLES:
    name = f"resampled, S={n_samples}"
    pz[name], components[name] = znn.ensemble_predict_resampled(
        trained_models,
        test_y1,
        ERRORS,
        n_samples,
        np.random.default_rng(NOISE_SEED),
        BANDS,
        REF_BAND,
        ids=ids,
        return_components=True,
    )

VERSIONS = list(pz)
z_point, width = {}, {}
for name in VERSIONS:
    z_point[name] = np.squeeze(pz[name].ancil["zmode"])
    width[name] = znn.predictions_mean_std(pz[name])[1].ravel()
    print(f"{name}: median p(z) width = {np.median(width[name]):.4f}")

## Where does the width come from?

For the largest S: median model and photometric parts of the width / (1 + z), in bins of i magnitude and true redshift, against the actual scatter of Δz (biweight). The photometric part should grow towards faint magnitudes, where the baseline (model part only) falls short of the actual scatter.

In [ ]:
redshift_bins = np.linspace(0, 2.5, 11)
imag_bins = np.linspace(18, 25.5, 11)


def binned_median(x, values, bins):
    idx = np.digitize(x, bins) - 1
    return np.array([np.median(values[idx == i]) if np.any(idx == i) else np.nan for i in range(len(bins) - 1)])


largest = f"resampled, S={max(N_SAMPLES)}"
stats_largest = znn.get_all_stats(
    Y_test_y1, z_point[largest], mag_i_test, save=False, redshift_bins=redshift_bins, imag_bins=imag_bins
)

fig, axarr = plt.subplots(1, 2, figsize=[13, 5])
for ax, x, bins, binned, xlabel in [
    (axarr[0], mag_i_test, imag_bins, stats_largest[2], "i magnitude"),
    (axarr[1], Y_test_y1, redshift_bins, stats_largest[1], "true redshift"),
]:
    centres = (bins[1:] + bins[:-1]) / 2
    for part, style in [("model", "-o"), ("photometric", "-s")]:
        ax.plot(centres, binned_median(x, components[largest][part] / (1 + Y_test_y1), bins), style, label=f"{part} part")
    ax.plot(centres, binned_median(x, width[largest] / (1 + Y_test_y1), bins), "-^", color="k", label="total width")
    ax.plot(centres, [s[2] for s in binned], "k--", label="actual scatter")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(r"$\sigma / (1+z)$")
    ax.set_yscale("log")
    ax.legend(fontsize=10)
plt.suptitle(largest, y=1.02)
plt.tight_layout()
plt.show()

## Point estimates

Δz = (z_phot − z_true) / (1 + z_true), using the p(z) mean as z_phot. For the resampled versions it is the mean over noisy photometry, which can shift slightly from the baseline.

In [ ]:
stats = {}
for name in VERSIONS:
    stats[name] = znn.get_all_stats(
        Y_test_y1, z_point[name], mag_i_test, save=False, redshift_bins=redshift_bins, imag_bins=imag_bins
    )
    znn.plot_stats(*stats[name], Y_test_y1, z_point[name], redshift_bins, imag_bins, mag_i_test)
    plt.suptitle(name, y=1.02)
    plt.show()

The largest-S version against the baseline: solid lines are the baseline, dashed lines the resampled version.

In [ ]:
display(Markdown(znn.stats_to_markdown(stats["baseline"][0], stats[largest][0], data_title=("baseline", largest))))
znn.compare_binned_stats(
    redshift_bins, imag_bins, stats["baseline"][1], stats["baseline"][2], stats[largest][1], stats[largest][2]
)
plt.suptitle(f"baseline (solid) vs {largest} (dashed)", y=1.02)
plt.show()

## p(z) calibration: PIT

PIT = CDF of each object's p(z) at its true redshift. For calibrated p(z) the PIT values are uniform: a flat histogram and a Q–Q curve on the diagonal. A U shape means p(z) too narrow, a hump means too wide. KS, CvM and Anderson–Darling measure the distance from uniform (lower is better). `outlier` is the fraction of PIT values in the extreme tails (< 0.0001 or > 0.9999).

In [ ]:
pit_values, pit_stats = {}, {}
for name in VERSIONS:
    pit = PIT(pz[name], Y_test_y1)
    pit_values[name] = pit.pit_samps
    pit_stats[name] = {
        "ks": pit.evaluate_PIT_KS().statistic,
        "CvM": pit.evaluate_PIT_CvM().statistic,
        "ksamp": pit.evaluate_PIT_anderson_ksamp().statistic,
        "outlier": np.mean((pit.pit_samps < 0.0001) | (pit.pit_samps > 0.9999)),
    }

fig, axarr = plt.subplots(1, 2, figsize=[13, 5])
q = np.linspace(0, 1, 101)
for name in VERSIONS:
    axarr[0].hist(pit_values[name], bins=50, range=(0, 1), density=True, histtype="step", lw=2, label=name)
    axarr[1].plot(q, np.quantile(pit_values[name], q), lw=2, label=name)
axarr[0].axhline(1, color="k", ls="--")
axarr[0].set_xlabel("PIT")
axarr[0].set_ylabel("density")
axarr[0].legend(fontsize=10)
axarr[1].plot([0, 1], [0, 1], "k--")
axarr[1].set_xlabel("uniform quantile")
axarr[1].set_ylabel("PIT quantile")
plt.tight_layout()
plt.show()

pd.DataFrame(pit_stats).round(4)

## Do the widths match the actual errors?

Left and middle: median p(z) width / (1 + z) in bins of i magnitude and true redshift, against the actual scatter of Δz (biweight, from `get_all_stats`). For calibrated p(z) the two should track each other. Right: residuals divided by the predicted width, which should follow a unit Gaussian.

In [ ]:
fig, axarr = plt.subplots(1, 3, figsize=[18, 5])
for k, name in enumerate(VERSIONS):
    rel_width = width[name] / (1 + Y_test_y1)
    for ax, x, bins, binned in [
        (axarr[0], mag_i_test, imag_bins, stats[name][2]),
        (axarr[1], Y_test_y1, redshift_bins, stats[name][1]),
    ]:
        centres = (bins[1:] + bins[:-1]) / 2
        ax.plot(centres, binned_median(x, rel_width, bins), "-o", color=f"C{k}", label=f"{name}: median width")
        ax.plot(centres, [s[2] for s in binned], "--", color=f"C{k}", label=f"{name}: actual scatter")
    axarr[2].hist(
        (z_point[name] - Y_test_y1) / width[name], bins=np.linspace(-6, 6, 121), density=True, histtype="step", lw=2, label=name
    )

zz = np.linspace(-6, 6, 200)
axarr[2].plot(zz, np.exp(-0.5 * zz**2) / np.sqrt(2 * np.pi), "k--", label="unit Gaussian")
axarr[0].set_xlabel("i magnitude")
axarr[1].set_xlabel("true redshift")
for ax in axarr[:2]:
    ax.set_ylabel(r"$\sigma / (1+z)$")
    ax.set_yscale("log")
    ax.legend(fontsize=8)
axarr[2].set_xlabel("(z_phot - z_true) / width")
axarr[2].legend(fontsize=9)
plt.tight_layout()
plt.show()

## Example p(z)

A few random test objects: each version's p(z), with the true redshift as a dashed line.

In [ ]:
rng = np.random.default_rng(1)
examples = rng.choice(len(Y_test_y1), size=8, replace=False)
z_grid = np.linspace(0, 3, 601)

fig, axarr = plt.subplots(2, 4, figsize=[18, 7])
for ax, i in zip(axarr.ravel(), examples):
    for k, name in enumerate(VERSIONS):
        ax.plot(z_grid, np.squeeze(pz[name][int(i)].pdf(z_grid)), color=f"C{k}", label=name)
    ax.axvline(Y_test_y1[i], color="k", ls="--")
    ax.set_xlim(max(0, Y_test_y1[i] - 0.6), Y_test_y1[i] + 0.6)
    ax.set_title(f"i = {mag_i_test[i]:.1f}, z = {Y_test_y1[i]:.2f}", fontsize=11)
axarr[0, 0].legend(fontsize=8)
for ax in axarr[1]:
    ax.set_xlabel("z")
plt.tight_layout()
plt.show()

## Summary

In [ ]:
summary = {}
for name in VERSIONS:
    bias, bias_err, scatter, outlier_rate, abs_outlier_rate = stats[name][0]
    summary[name] = {
        "bias": bias,
        "scatter": scatter,
        "outlier rate (3 sigma)": outlier_rate,
        "outlier rate (|dz| > 0.2)": abs_outlier_rate,
        "median p(z) width": np.median(width[name]),
        **{f"PIT {k}": v for k, v in pit_stats[name].items()},
    }
pd.DataFrame(summary).round(4)